# 02 Feature activations at "="

Step 1 of the label plan (preregistration section 3): collect every transcoder feature's activation at the "=" position.

Setup: Accelerator GPU T4 x2, Internet on, secret `HF_TOKEN`.

**New session:** run cells 1 to 4 in order. If a later cell complains about the `transformers` version, restart once and continue from cell 2.

**After pushing new code:** run `update()` (cell 3). No restart, and the model stays loaded. Use modules as `activations.eq_activations(...)`, not `from ... import eq_activations`, so the reload reaches them.

In [ ]:
# 1. get the code and the libraries
!rm -rf helix_addition && git clone https://github.com/amark-23/helix_addition.git
!pip install -q git+https://github.com/safety-research/circuit-tracer@v0.5.3 pytest

In [ ]:
# 2. make helixadd importable (no install needed) and log in to Hugging Face
import sys
sys.path.insert(0, "helix_addition/src")

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
# 3. update(): pull the latest code and reload every helixadd module in place
import importlib, subprocess

def update():
    print(subprocess.run(["git", "-C", "helix_addition", "pull"], capture_output=True, text=True).stdout)
    names = sorted(m for m in sys.modules if m.startswith("helixadd"))
    # modules import each other, so a reload can fail until its dependencies are reloaded;
    # retry the failed ones until everything reloads
    for _ in range(len(names)):
        failed = []
        for name in names:
            try:
                importlib.reload(sys.modules[name])
            except Exception as e:
                failed.append((name, e))
        if not failed:
            print("reloaded", names)
            return
        names = [n for n, _ in failed]
    raise RuntimeError(f"could not reload: {failed}")

!cd helix_addition && python -m pytest -q

In [ ]:
# 4. load the model (once per session)
from helixadd import model as hmodel, prompts, activations
model = hmodel.load_model()

## How many features fire at "="?

In [ ]:
_, acts = model.get_activations(prompts.make_prompt(36, 59))
print(acts.shape)
eq = acts[:, -1, :]                     # every layer, last position ("="), every feature
print((eq > 0).sum(dim=1))              # active features per layer
print("total:", (eq > 0).sum().item())

## eq_activations on one prompt

In [ ]:
layers, features, values = activations.eq_activations(model, prompts.make_prompt(36, 59))
print(len(layers), layers[:5], features[:5], values[:5])

## Test: 20 prompts

Checks that the loop and the save work before the 30-minute run. Writes to a separate test folder.

In [ ]:
test = activations.collect(model, "helix_addition/results/eq_test", n=20)
print({k: v.shape for k, v in test.items()})
print("entries per prompt:", len(test["layer"]) / 20)

## Full run: all 9,992 prompts (about 30 minutes)

Run as Save Version, Save & Run All. Output: `helix_addition/results/eq_activations/`.

In [ ]:
data = activations.collect(model, "helix_addition/results/eq_activations")
print({k: v.shape for k, v in data.items()})
print("entries per prompt:", len(data["layer"]) / len(data["a"]))